In [7]:
import numpy as np

In [1]:
text = 'you say goodbye and I say hello.'
text = text.lower()
text = text.replace('.', ' .')
text

'you say goodbye and i say hello .'

In [2]:
words = text.split(' ') # 단어 단위로 분할
words

['you', 'say', 'goodbye', 'and', 'i', 'say', 'hello', '.']

In [9]:
# 단어 ID와 단어 사전 생성

words_to_id = {}
id_to_words = {}

for word in words:
    if word not in words_to_id:
        new_id = len(words_to_id) # 새로운 단어 추가, ID는 현재의 단어 수
        words_to_id[word] = new_id
        id_to_words[new_id] = word
        
words_to_id
# id_to_words

{'you': 0, 'say': 1, 'goodbye': 2, 'and': 3, 'i': 4, 'hello': 5, '.': 6}

In [10]:
words_to_id['hello']

5

In [8]:
corpus = [words_to_id[w] for w in words]
corpus = np.array(corpus)
corpus

array([0, 1, 2, 3, 4, 1, 5, 6])

In [11]:
# 말뭉치 전처리 함수
def preprocess(text):
    text = text.lower()
    text = text.replace('.', ' .')
    words = text.split(' ')
    
    word_to_id = {}
    id_to_word = {}
    for word in words:
        if word not in word_to_id:
            new_id = len(word_to_id)
            word_to_id[word] = new_id
            id_to_word[new_id] = word
            
    corpus = np.array([word_to_id[w] for w in words])
    
    return corpus, word_to_id, id_to_word

In [12]:
text = 'You say goodbye and I say hello.'
corpus, word_to_id, id_to_word = preprocess(text)

In [14]:
print(corpus)
print(word_to_id)
print(id_to_word)

[0 1 2 3 4 1 5 6]
{'you': 0, 'say': 1, 'goodbye': 2, 'and': 3, 'i': 4, 'hello': 5, '.': 6}
{0: 'you', 1: 'say', 2: 'goodbye', 3: 'and', 4: 'i', 5: 'hello', 6: '.'}


### 동시발생 행렬, 분포 가설에 기초해 단어를 벡터로 구현  

In [15]:
# 동시발생 행렬

c = np.array([[0, 1, 0, 0, 0, 0, 0],
              [1, 0, 1, 0, 1, 1, 0],
              [0, 1, 0, 1, 0, 0, 0],
              [0, 0, 1, 0, 1, 0, 0],
              [0, 1, 0, 1, 0, 0, 0],
              [0, 1, 0, 0, 0, 0, 1],
              [0, 0, 0, 0, 0, 1, 0]], dtype=np.int32)

In [17]:
print(c[0]) # ID가 0인 단어의 벡터 표현
print(c[word_to_id['goodbye']]) # goodbye의 벡터 표현


[0 1 0 0 0 0 0]
[0 1 0 1 0 0 0]


In [25]:
# 윈도우 크기가 1인 경우의 동시발생 행렬 생성 함수

def create_co_matrix(corpus, vocab_size, window_size=1): # corpus: 단어 ID 리스트, vocab_size: 어휘 수, window_size: 윈도우 크기 (기본값 1)
  
  corpus_size = len(corpus) # 말뭉치 크기
  co_matrix = np.zeros((vocab_size, vocab_size), dtype=np.int32) # 동시발생 행렬 0으로 초기화
    
  for idx, word_id in enumerate(corpus): # idx는 인덱스이고, word_id는 단어 ID이다. idx는 순서 번호를 나타낸다. word_id는 단어 ID를 나타낸다.
    for i in range(1, window_size + 1): # 윈도우 크기가 1인 경우, 좌우 한 단어씩만 살펴본다.
      left_idx = idx - i                # 타깃 단어의 왼쪽에 있는 단어의 인덱스
      right_idx = idx + i               # 타깃 단어의 오른쪽에 있는 단어의 인덱스
            
      if left_idx >= 0:
        left_word_id = corpus[left_idx]
        co_matrix[word_id, left_word_id] += 1  # word_id: 행, left_word_id: 열
                
      if right_idx < corpus_size:
        right_word_id = corpus[right_idx]
        co_matrix[word_id, right_word_id] += 1
                
  return co_matrix



In [34]:
# 예시 코드

# 먼저 말뭉치 전처리
text = 'you say goodbye and I say hello.'
corpus, word_to_id, id_to_word = preprocess(text)

print(corpus)
print(word_to_id)
print(id_to_word)

# 동시발생 행렬 생성
vocab_size = len(id_to_word) # word_to_id를 넣어도 됨
C = create_co_matrix(corpus, vocab_size)
print(C)

[0 1 2 3 4 1 5 6]
{'you': 0, 'say': 1, 'goodbye': 2, 'and': 3, 'i': 4, 'hello': 5, '.': 6}
{0: 'you', 1: 'say', 2: 'goodbye', 3: 'and', 4: 'i', 5: 'hello', 6: '.'}
[[0 1 0 0 0 0 0]
 [1 0 1 0 1 1 0]
 [0 1 0 1 0 0 0]
 [0 0 1 0 1 0 0]
 [0 1 0 1 0 0 0]
 [0 1 0 0 0 0 1]
 [0 0 0 0 0 1 0]]


In [31]:
for idx, word_id in enumerate(corpus):
  print(idx, word_id)

0 0
1 1
2 2
3 3
4 4
5 1
6 5
7 6


### 벡터 간 유사도 측정  

In [ ]:
# 코사인 유사도 함수
def cosine_similarity(x,y, eps=1e-8): # x 벡터, y 벡터, eps는 인수로 제로 벡터가 들어오면 0으로 나누는 사태를 막기 위한 작은 값
  nx = x / np.sqrt(np.sum(x**2) + eps) # x의 정규화
  ny = y / np.sqrt(np.sum(y**2) + eps) # y의 정규화
  return np.dot(nx, ny) # 정규화된 x와 y의 내적
  

In [35]:
x = np.array([100, -20, 2])
y = np.array([-100, 50, 3])

nx = x / np.sqrt(np.sum(x**2))
ny = y / np.sqrt(np.sum(y**2))
print(nx)
print(ny)

[ 0.98039216 -0.19607843  0.01960784]
[-0.89410537  0.44705269  0.02682316]
